# Waste AI — Entraînement amélioré (v2)
**TrashNet + TACO + Data Augmentation forte + Fine-tuning complet**

Objectif : améliorer la robustesse sur des images réelles (fond complexe, mauvaise lumière, objets multiples)

Lancez sur Google Colab avec GPU T4.

## 1. Installation & imports

In [ ]:
!pip install torch torchvision matplotlib scikit-learn pycocotools -q
!pip install albumentations -q

import torch
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, ConcatDataset, random_split, Dataset
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from PIL import Image
import os, json, shutil

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
assert DEVICE == 'cuda', 'Active le GPU : Exécution > Modifier le type > GPU T4'

## 2. Téléchargement des datasets

In [ ]:
# --- TrashNet ---
!wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip
!unzip -q dataset-resized.zip -d trashnet/
print('TrashNet téléchargé')

In [ ]:
# --- TACO ---
!git clone --quiet https://github.com/pedropro/TACO.git
!cd TACO && pip install -r requirements.txt -q
!cd TACO && python download.py
print('TACO téléchargé')

## 3. Mapping TACO → catégories Waste AI

In [ ]:
# Correspondance entre catégories TACO et nos 6 classes
TACO_TO_CLASS = {
    # Plastique
    'Plastic bottle': 'plastic',
    'Plastic bag & wrapper': 'plastic',
    'Plastic container': 'plastic',
    'Plastic cup': 'plastic',
    'Plastic lid': 'plastic',
    'Plastic straw': 'plastic',
    'Plastic utensils': 'plastic',
    'Six pack rings': 'plastic',
    'Styrofoam piece': 'plastic',
    # Papier / Carton
    'Paper': 'paper',
    'Paper bag': 'paper',
    'Newspaper': 'paper',
    'Cardboard': 'cardboard',
    'Carton': 'cardboard',
    'Drink carton': 'cardboard',
    # Verre
    'Glass bottle': 'glass',
    'Broken glass': 'glass',
    'Glass jar': 'glass',
    # Métal
    'Aluminium foil': 'metal',
    'Metal bottle cap': 'metal',
    'Drink can': 'metal',
    'Food can': 'metal',
    'Pop tab': 'metal',
    'Scrap metal': 'metal',
    # Résidus
    'Cigarette': 'trash',
    'Rope & strings': 'trash',
    'Shoe': 'trash',
    'Unlabeled litter': 'trash',
}

CLASS_NAMES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

# Extraire les images TACO et les organiser par classe
TACO_OUT = 'taco_organized'
for cls in CLASS_NAMES:
    os.makedirs(f'{TACO_OUT}/{cls}', exist_ok=True)

with open('TACO/data/annotations.json') as f:
    taco_data = json.load(f)

# Index images
id_to_file = {img['id']: img['file_name'] for img in taco_data['images']}
cat_id_to_name = {cat['id']: cat['name'] for cat in taco_data['categories']}

count = 0
for ann in taco_data['annotations']:
    cat_name = cat_id_to_name.get(ann['category_id'], '')
    target_class = TACO_TO_CLASS.get(cat_name)
    if target_class is None:
        continue
    src = f"TACO/data/{id_to_file[ann['image_id']]}"
    dst = f"{TACO_OUT}/{target_class}/{ann['image_id']}_{ann['id']}.jpg"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)
        count += 1

print(f'TACO : {count} images organisées')
for cls in CLASS_NAMES:
    n = len(os.listdir(f'{TACO_OUT}/{cls}'))
    print(f'  {cls}: {n} images')

## 4. Dataset custom avec Albumentations (augmentation forte)

In [ ]:
class WasteDataset(Dataset):
    """Dataset qui charge des images depuis un dossier ImageFolder-style avec albumentations."""
    def __init__(self, root, transform=None):
        self.data = datasets.ImageFolder(root)
        self.transform = transform
        self.classes = self.data.classes

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data.samples[idx]
        image = np.array(Image.open(path).convert('RGB'))
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label


# Augmentation forte pour entraînement (robustesse contexte réel)
train_transform = A.Compose([
    A.RandomResizedCrop(224, 224, scale=(0.5, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=45, p=0.5),
    A.OneOf([
        A.GaussNoise(var_limit=(10, 50)),
        A.GaussianBlur(blur_limit=3),
        A.MotionBlur(blur_limit=3),
    ], p=0.4),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
        A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30),
        A.CLAHE(clip_limit=4),
    ], p=0.5),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

print('Transforms définis')

## 5. Chargement et fusion des datasets

In [ ]:
trashnet_ds = WasteDataset('trashnet/dataset-resized', transform=train_transform)
taco_ds = WasteDataset(TACO_OUT, transform=train_transform)

print(f'TrashNet : {len(trashnet_ds)} images | classes : {trashnet_ds.classes}')
print(f'TACO     : {len(taco_ds)} images | classes : {taco_ds.classes}')

# Vérifier que les classes sont identiques
assert trashnet_ds.classes == taco_ds.classes, \
    f'Classes différentes !\nTrashNet: {trashnet_ds.classes}\nTACO: {taco_ds.classes}'

full_dataset = ConcatDataset([trashnet_ds, taco_ds])
print(f'Dataset combiné : {len(full_dataset)} images')

# Split 80/20
n_train = int(0.8 * len(full_dataset))
n_val = len(full_dataset) - n_train
train_set, val_set = random_split(full_dataset, [n_train, n_val],
                                   generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 32
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'Train : {n_train} | Val : {n_val}')

## 6. Modèle — Fine-tuning complet de MobileNetV3

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)
EPOCHS = 20

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, NUM_CLASSES)
model = model.to(DEVICE)

# Phase 1 : entraîner seulement la tête (5 epochs)
for param in model.features.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(model.classifier.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

print(f'Modèle prêt — {NUM_CLASSES} classes')
print('Phase 1 : entraînement de la tête uniquement (5 epochs)')

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total * 100


history = {'train_loss': [], 'val_acc': []}

# Phase 1 — tête seulement
for epoch in range(5):
    train_loss, _ = run_epoch(train_loader, train=True)
    _, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    print(f'[Phase 1] Epoch {epoch+1}/5 | Loss: {train_loss:.4f} | Val Acc: {val_acc:.1f}%')

In [ ]:
# Phase 2 : débloquer toutes les couches et fine-tuner avec lr faible
print('\nPhase 2 : fine-tuning complet (15 epochs, lr=1e-4)')
for param in model.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

best_acc = 0
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(15):
    train_loss, _ = run_epoch(train_loader, train=True)
    _, val_acc = run_epoch(val_loader, train=False)
    scheduler.step()
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    print(f'[Phase 2] Epoch {epoch+1}/15 | Loss: {train_loss:.4f} | Val Acc: {val_acc:.1f}%')

    # Sauvegarder le meilleur modèle
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'checkpoints/waste_ai_v2.pt')
        print(f'  -> Meilleur modèle sauvegardé ({val_acc:.1f}%)')

print(f'\nMeilleure accuracy : {best_acc:.1f}%')

## 7. Évaluation — Matrice de confusion

In [ ]:
# Charger le meilleur modèle
model.load_state_dict(torch.load('checkpoints/waste_ai_v2.pt'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs).argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

# Rapport de classification
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Matrice de confusion — Waste AI v2')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.tight_layout()
plt.savefig('checkpoints/confusion_matrix.png', dpi=150)
plt.show()

## 8. Courbes d'entraînement

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], color='#e74c3c')
ax1.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history['val_acc'], color='#2ecc71')
ax2.axvline(x=4, color='gray', linestyle='--', label='Début phase 2')
ax2.set_title('Val Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.suptitle('Waste AI v2 — TrashNet + TACO', fontsize=14)
plt.tight_layout()
plt.savefig('checkpoints/training_curves_v2.png', dpi=150)
plt.show()

## 9. Télécharger le modèle

In [ ]:
from google.colab import files

# Télécharger le modèle amélioré
files.download('checkpoints/waste_ai_v2.pt')
files.download('checkpoints/confusion_matrix.png')
files.download('checkpoints/training_curves_v2.png')

print('Téléchargements lancés !')
print('-> Remplace waste_ai.pt par waste_ai_v2.pt dans model/checkpoints/')
print('   ET mets à jour MODEL_PATH dans api/main.py')